# 31 · Load: Anthropic AI exposure (observed exposure) → `staging`

**Source card:** `anthropic_ai_exposure.md` (§6 issues, §7 staging tables). Decision: D7 revised: this is the main AI score; Eloundou β (nb 28) stays beside it.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_anthropic__job_exposure` | SOC 2018 detailed job | 756 |
| `stg_anthropic__task_penetration` | O\*NET task statement (text) | 17,992 |

Anthropic Economic Index, "Labor market impacts of AI" (Massenkoff & McCrory, Mar 2026), Hugging Face commit `4e5f69f`, CC-BY. Observed exposure starts from Eloundou's task labels and keeps only tasks seen in real work use of Claude (automated use counts fully, assisted half), weighted by time spent on each task. Penetration is the share of a task's exposure seen in that use. The task file has no job code or task ID, only the task text; it joins to jobs through Eloundou's task table (§2).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import STAGE_DIR, connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 90, "display.width", 220)
NB = "31-load-anthropic-ai-exposure.ipynb"
conn = connect()

F = raw_files("anthropic_ai_exposure")
def path(name: str):
    row = F[F.file.str.endswith("/" + name)].iloc[-1]
    return row.path, row.file

p_job, f_job = path("job_exposure.csv")
p_task, f_task = path("task_penetration.csv")
job = pd.read_csv(p_job, dtype=str, keep_default_na=False)
task = pd.read_csv(p_task, dtype=str, keep_default_na=False)
print("job_exposure:", job.shape, "| task_penetration:", task.shape)

job_exposure: (756, 3) | task_penetration: (17998, 2)


## 1. Jobs: codes, scores, titles

Every code should be a SOC 2018 detailed job (checked against the BLS SOC structure staged in nb 24). Scores must be numbers in 0–1. Zero is a real value ("not seen yet" band in D7), not a blank.

In [2]:
assert not job.occ_code.duplicated().any()
assert job.occ_code.str.fullmatch(r"\d\d-\d{4}").all()
assert (job.title.str.strip() != "").all() and (job.title == job.title.str.strip()).all()
job["observed_exposure"] = pd.to_numeric(job.observed_exposure, errors="raise").astype("float64")
assert job.observed_exposure.between(0, 1).all()

soc = pd.read_parquet(STAGE_DIR / "stg_bls__soc_2018_structure.parquet")
lvl = job.occ_code.map(soc.set_index("soc_code").level)
print("SOC 2018 level of each code:", lvl.fillna("not in SOC 2018").value_counts().to_dict())
assert (lvl == "detailed").all()
title_diff = job.merge(soc[["soc_code", "soc_title"]], left_on="occ_code", right_on="soc_code")
title_diff = title_diff[title_diff.title.str.lower() != title_diff.soc_title.str.lower()]
print(f"Titles that differ from the BLS SOC title: {len(title_diff)} (kept as published)")
display(title_diff[["occ_code", "title", "soc_title"]].head(8))

b = pd.cut(job.observed_exposure, [-1, 1e-12, 0.10, 0.30, 2], right=False,
           labels=["0 (not seen yet)", "low <0.10", "medium 0.10-0.30", "high >=0.30"])
print("Jobs per D7 band (unweighted):", b.value_counts().sort_index().to_dict())
display(job.observed_exposure.describe().round(4).to_frame().T)
display(job.nlargest(8, "observed_exposure"))

SOC 2018 level of each code: {'detailed': 756}
Titles that differ from the BLS SOC title: 1 (kept as published)


,occ_code,title,soc_title
367,33-3051,Police and Sheriff's Patrol Officers,Police and Sheriff’s Patrol Officers


Jobs per D7 band (unweighted): {'0 (not seen yet)': 411, 'low <0.10': 162, 'medium 0.10-0.30': 116, 'high >=0.30': 67}


,count,mean,std,min,25%,50%,75%,max
observed_exposure,756.0,0.077,0.1332,0.0,0.0,0.0,0.0957,0.7451


,occ_code,title,observed_exposure
73,15-1251,Computer Programmers,0.7451
461,43-4051,Customer Service Representatives,0.7011
488,43-9021,Data Entry Keyers,0.6707
330,29-2072,Medical Records Specialists,0.6674
49,13-1161,Market Research Analysts and Marketing Specialists,0.6483
351,31-9094,Medical Transcriptionists,0.6365
439,41-4012,"Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Pr...",0.6279
71,15-1243,Database Architects,0.5787


## 2. Tasks: duplicates and the link to jobs

The file is one row per job × task with the job dropped, so a task shared by several jobs repeats. Repeats are identical (same text, same penetration) and are dropped; the key is then the task text. One task appears in two capitalizations (`Web pages` / `web pages`); both are kept, since each matches its own O\*NET row.

In [3]:
task["penetration"] = pd.to_numeric(task.penetration, errors="raise").astype("float64")
assert task.penetration.between(0, 1).all()
assert (task.task == task.task.str.strip()).all() and (task.task != "").all()
rep = task[task.task.duplicated(keep=False)].groupby("task").penetration.agg(["nunique", "count"])
display(rep)
assert (rep["nunique"] == 1).all(), "a repeated task has two different penetration values"
t = task.drop_duplicates().reset_index(drop=True)
log("drop exact repeats of a task", len(task), len(t))
assert not t.task.duplicated().any()
print(f"Tasks with penetration > 0: {(t.penetration > 0).sum():,} of {len(t):,} ({(t.penetration > 0).mean():.1%})")

el = pd.read_parquet(STAGE_DIR / "stg_eloundou__task_exposure.parquet")
m = t.task.isin(set(el.task))
print(f"Task text found in Eloundou's task table (stg_eloundou__task_exposure): {m.sum():,} of {len(t):,} ({m.mean():.1%})")
assert m.all(), "every Anthropic task should link to an O*NET job through Eloundou"
jobs_per_task = el[el.task.isin(set(t.task))].groupby("task").onet_soc_code.nunique()
print("O*NET jobs per Anthropic task:", jobs_per_task.value_counts().sort_index().to_dict())

,nunique,count
task,,
"Write interesting and effective press releases, prepare information for media kits, and develop and maintain company internet or intranet Web pages.",1,4
"Write interesting and effective press releases, prepare information for media kits, and develop and maintain company internet or intranet web pages.",1,4


  drop exact repeats of a task: 17,998 -> 17,992 rows (-6)
Tasks with penetration > 0: 1,348 of 17,992 (7.5%)
Task text found in Eloundou's task table (stg_eloundou__task_exposure): 17,992 of 17,992 (100.0%)


O*NET jobs per Anthropic task: {1: 17585, 2: 298, 3: 42, 4: 11, 5: 8, 6: 5, 7: 7, 8: 2, 9: 7, 10: 2, 11: 1, 12: 2, 13: 1, 18: 1, 21: 1, 22: 1, 29: 2, 31: 1, 32: 3, 33: 6, 34: 6}


## 3. Tie-out

Anthropic does not publish a total to tie to. Checks instead: (a) every row in each file is kept (job 756 → 756; tasks 17,998 → 17,992 after 6 exact repeats); (b) the job score agrees with Eloundou β about as well as D7 recorded (0.65 Pearson, 0.72 rank), since observed exposure is built from those labels. Here the comparison is one row per SOC job with β averaged over its O\*NET jobs, so it lands a little lower (0.64 / 0.70).

In [4]:
ej = pd.read_parquet(STAGE_DIR / "stg_eloundou__job_exposure.parquet").groupby("soc_code").beta_gpt4.mean()
c = job.set_index("occ_code").observed_exposure.to_frame().join(ej, how="inner")
r, rs = c.corr().iloc[0, 1], c.corr("spearman").iloc[0, 1]
print(f"Jobs in both: {len(c)} of {len(job)}; Pearson {r:.2f}, rank {rs:.2f} (D7 recorded 0.65 / 0.72)")
assert abs(r - 0.65) < 0.05 and abs(rs - 0.72) < 0.05, "agreement with Eloundou moved; recheck before staging"
above = c[c.observed_exposure > c.beta_gpt4 + 1e-9]
print(f"Jobs scoring above their Eloundou beta: {len(above)}. Not an error: beta counts tasks equally (core 1, supplemental 0.5),\n"
      "observed exposure weights tasks by time spent, so a job whose exposed tasks fill most of the day can score higher.")
display(above.join(job.set_index("occ_code").title).round(3))

Jobs in both: 756 of 756; Pearson 0.64, rank 0.70 (D7 recorded 0.65 / 0.72)
Jobs scoring above their Eloundou beta: 7. Not an error: beta counts tasks equally (core 1, supplemental 0.5),
observed exposure weights tasks by time spent, so a job whose exposed tasks fill most of the day can score higher.


,observed_exposure,beta_gpt4,title
occ_code,,,
13-1161,0.648,0.525,Market Research Analysts and Marketing Specialists
13-2051,0.572,0.462,Financial and Investment Analysts
29-2072,0.667,0.618,Medical Records Specialists
41-4012,0.628,0.621,"Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Pr..."
43-4051,0.701,0.568,Customer Service Representatives
47-2221,0.049,0.029,Structural Iron and Steel Workers
51-2011,0.049,0.025,"Aircraft Structure, Surfaces, Rigging, and Systems Assemblers"


## 4. Stage

In [5]:
stg_job = (job.rename(columns={"occ_code": "soc_code"})[["soc_code", "title", "observed_exposure"]]
           .assign(source_file=f_job).sort_values("soc_code").reset_index(drop=True))
display(stage(stg_job, "stg_anthropic__job_exposure", ["soc_code"], expected_rows=756, source_key="anthropic_ai_exposure",
              notebook=NB, conn=conn,
              comment="Anthropic Economic Index observed exposure per SOC 2018 detailed job (Massenkoff & McCrory, Mar 2026), 0-1. Main AI score (D7 revised)."))

stg_task = t.assign(source_file=f_task).sort_values("task").reset_index(drop=True)
display(stage(stg_task, "stg_anthropic__task_penetration", ["task"], expected_rows=17_992, source_key="anthropic_ai_exposure",
              notebook=NB, conn=conn,
              comment="Anthropic Economic Index task penetration: share of each O*NET task's exposure seen in work use of Claude. Keyed by task text; joins to jobs via stg_eloundou__task_exposure.task."))
if conn is not None:
    conn.close()

  stg_anthropic__job_exposure: 756 rows x 4 cols, key ['soc_code'] unique | local data/interim/staging/stg_anthropic__job_exposure.parquet | DB: 756


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_anthropic__job_exposure,756,4,soc_code,anthropic_ai_exposure,31-load-anthropic-ai-exposure.ipynb,756,2026-10-14T21:57:52+00:00


  stg_anthropic__task_penetration: 17,992 rows x 3 cols, key ['task'] unique | local data/interim/staging/stg_anthropic__task_penetration.parquet | DB: 17992


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_anthropic__task_penetration,17992,3,task,anthropic_ai_exposure,31-load-anthropic-ai-exposure.ipynb,17992,2026-10-14T21:57:52+00:00
